**Customer Satisfaction Prediction using Machine Learning Project**

In [ ]:
#importing important Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [ ]:
# Load the dataset
data=pd.read_csv('/content/customer_support_tickets.csv')

In [ ]:
# Find the rows of a dataset
print(data.head())

In [ ]:
#Display basic info about the dataset
print(data.info())

**Data** **Preprocessing**

In [ ]:
# Find the missing values

data.isnull().sum()

In [ ]:
# dropping null values
data.dropna(inplace=True)

In [ ]:
#Rechecking null values after dropping
data.isnull().sum()

In [ ]:
#Encoding Categorical variables

label_encoders={}
for column in data.select_dtypes(include=['object']).columns:
  label_encoders[column]=LabelEncoder()

  data[column]=label_encoders[column].fit_transform(data[column])


In [ ]:
data.shape

In [ ]:
#Define Features and target variable

data.columns
x=data.drop(['Ticket ID','Customer Name','Customer Email','Ticket Subject','Ticket Description','Customer Satisfaction Rating'],axis=1)
y=data['Customer Satisfaction Rating']


In [ ]:
#splitting the dataset

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)
print(x_train)
print(x_test)
print(y_train)
print(y_test)

In [ ]:
# Feature Scaling

scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)   #learn+transfrom
x_test=scaler.transform(x_test)         #only transform

In [ ]:
#Model Building

rfc=RandomForestClassifier(random_state=42)
rfc=RandomForestClassifier(n_estimators=100,random_state=42)     #controls trees

#Train the model
rfc.fit(x_train,y_train)

#Predict on the test set
y_pred=rfc.predict(x_test)

In [ ]:
# model Evalution
print("Accuracy:",accuracy_score(y_test,y_pred))                                # Accuracy Score
print("\nclassification Report:\n",classification_report(y_test,y_pred))       #classification Report
print("\nconfusion Matrix:\n",confusion_matrix(y_test,y_pred))                 #Confusion Matrix

In [ ]:
#Feature Importance

feature_importances=pd.Series(rfc.feature_importances_, index=x.columns)
top10=feature_importances.sort_values(ascending=False).head(10)

print(top10)


**Observation:**

Time to Resolution has the highest impact on satisfaction

In [ ]:
# Visualization of results

import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
feature_importances.nlargest(10).plot(kind='bar')
plt.title('Top 10 Feature Importance')
plt.xlabel('Features')
plt.ylabel('Important Score')
plt.xticks(rotation=45)

plt.show()

**Observation :**

Resolution efficency significantly impacts customer satisfcation levels

In [ ]:
#Confusion Matrix
#This is shows correct predictions and wrong predictions

cm=confusion_matrix(y_test,y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm,annot=True,fmt='.0f',cmap='Blues') #float number 0 digits after decimal(displays them like integer)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

**Observation: Confusion Matrix **

The Confusion matrix shows the performance of the RandomForestClassifer across 5 customer satisfaction classes(0-4)

1.The diagonal value represent correct predictions.

2.The diagonal values (40,30,29,30,30) are not significantly higher than other values in each row.

3.This indicates that the model struggles to clearly distinguish between differet satisfaction levels.

There is noticeable confusion between classes:
1.Class 1 is frequently predicted as class 2.
2.Class 2 is often misclassified  as class 0.

predictions are distributed across multiple classes rather than being concentrated on the correct class.

Overall the model shows moderate to low Classification performance,as misclassifications are relatively high compared to correct predictions.


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

**Observation:**

1.Overall Accuracy:

Overall accuracy model is 19%
Since this is a 5-class classificaton problem, random guessing would give around 20% accuracy.
This means the model is performing close to random prediction.

2.Precision:

precision ranges between 0.17 and 0.22.
when the model predicts a satisfaction class, it is correctly only about 17-22% of the time.
The model does not confidently separate the classes.

3.Recall

Recall ranges between 0.17 and 0.24 of actual class instances.
many samples are classified.

4.F1-Scores

F1 Scores values are around 0.17-0.23

The balance between precision and recall is low.
Overall model performance is weak across  all classes.

5.Class Distribution:

support values are nearly equal:
168,174,175,162,152

The dataset is fairly balanced.
Poor performance is not due to class imbalance.
The issuse is likely feature quality or model tuning.

In [ ]:
#Improvement:
#Increase Number of trees

from sklearn.ensemble import RandomForestClassifier

rfc=RandomForestClassifier(n_estimators=300,max_depth=10,class_weight='balanced',random_state=42)
rfc.fit(x_train,y_train)

#Predict the model
y_pred=rfc.predict(x_test)

print(classification_report(y_test,y_pred))

**Observation:**

Class 1 precision improved (0.22-0.24).

class 3 recall improved.

overall macro average improved slightly(0.19-0.20).

Precision, recall and F1-Scores remain around 0.17-0.24,indicating weak class separability.

So this suggests that hyperparameter tuning alone is insufficient and that the feature set may contain strong predictive patterns.